# Evaluate Qwen2.5-VL on IAM Handwriting Dataset
Benchmarks CER and WER on [Teklia/IAM-line](https://huggingface.co/datasets/Teklia/IAM-line) (all splits).

> Runtime → Change runtime type → **T4 GPU**

In [5]:
!pip install -q jiwer datasets transformers torch torchvision qwen-vl-utils accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 70.5 MB/s eta 0:00:00:00:0100:01


In [2]:
# Mount Drive and cache HuggingFace weights so they don't re-download each session
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
print("HF cache:", os.environ["HF_HOME"])

Mounted at /content/drive
HF cache: /content/drive/MyDrive/cogs181/hf_cache


In [3]:
%%writefile evaluate.py
"""
Evaluate Qwen2.5-VL on IAM Handwriting Dataset

Loads the Teklia/IAM-line dataset from HuggingFace, runs each image through
the Qwen2.5-VL pipeline, and computes CER and WER against ground truth.
"""

import argparse
import csv
import time
from pathlib import Path

import torch
from jiwer import cer, wer


def main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate Qwen2.5-VL on IAM handwriting")
    parser.add_argument("--model_name", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--device", choices=["auto", "cpu", "cuda", "mps"], default="auto")
    parser.add_argument("--num_samples", type=int, default=100, help="Number of samples to evaluate (0 = all)")
    parser.add_argument("--output_dir", default="./Evaluation")
    parser.add_argument("--max_new_tokens", type=int, default=128)
    parser.add_argument(
        "--prompt",
        default="Transcribe the handwritten text in this image exactly. Output only the text, nothing else.",
    )
    args = parser.parse_args()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = args.device
    if device == "auto":
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
    print(f"Using device: {device}")

    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

    print(f"Loading model: {args.model_name}")
    t_load = time.time()
    processor = AutoProcessor.from_pretrained(args.model_name)

    if device == "cuda":
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            args.model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            ignore_mismatched_sizes=True,
        )
    else:
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            args.model_name,
            torch_dtype=torch.float32,
            ignore_mismatched_sizes=True,
        )
        model = model.to(device)
    model.eval()
    print(f"Model loaded in {time.time() - t_load:.1f}s")

    from datasets import load_dataset, concatenate_datasets

    print("Loading Teklia/IAM-line dataset (all splits) ...")
    ds_dict = load_dataset("Teklia/IAM-line")
    dataset = concatenate_datasets(list(ds_dict.values()))
    print(f"Total samples in dataset: {len(dataset)}")

    if args.num_samples > 0:
        dataset = dataset.select(range(min(args.num_samples, len(dataset))))
    print(f"Evaluating on {len(dataset)} samples")

    predictions = []
    references = []
    csv_path = output_dir / "eval_results.csv"

    t_start = time.time()
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["index", "ground_truth", "prediction", "sample_wer", "sample_cer"])
        writer.writeheader()

        for idx, sample in enumerate(dataset):
            image = sample["image"].convert("RGB")
            ground_truth = sample["text"].strip()

            if not ground_truth:
                continue

            messages = [{"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": args.prompt},
            ]}]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
            inputs = inputs.to(device)

            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=args.max_new_tokens)

            trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
            prediction = processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0].strip()

            sample_wer = wer(ground_truth, prediction) if ground_truth else 0.0
            sample_cer = cer(ground_truth, prediction) if ground_truth else 0.0

            predictions.append(prediction)
            references.append(ground_truth)

            writer.writerow({
                "index": idx,
                "ground_truth": ground_truth,
                "prediction": prediction,
                "sample_wer": f"{sample_wer:.4f}",
                "sample_cer": f"{sample_cer:.4f}",
            })

            if (idx + 1) % 10 == 0 or idx == 0:
                print(f"  [{idx+1}/{len(dataset)}] GT: {ground_truth[:50]}  |  PRED: {prediction[:50]}")

    elapsed = time.time() - t_start

    total_wer = wer(references, predictions)
    total_cer = cer(references, predictions)

    print("\n" + "=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"Model          : {args.model_name}")
    print(f"Samples        : {len(references)}")
    print(f"WER            : {total_wer:.4f} ({total_wer*100:.2f}%)")
    print(f"CER            : {total_cer:.4f} ({total_cer*100:.2f}%)")
    print(f"Time           : {elapsed:.1f}s ({elapsed/max(len(references),1):.2f}s/sample)")
    print(f"Results CSV    : {csv_path}")
    print("=" * 50)

    summary_path = output_dir / "eval_summary.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"Model: {args.model_name}\n")
        f.write(f"Samples: {len(references)}\n")
        f.write(f"WER: {total_wer:.4f} ({total_wer*100:.2f}%)\n")
        f.write(f"CER: {total_cer:.4f} ({total_cer*100:.2f}%)\n")
        f.write(f"Time: {elapsed:.1f}s\n")
    print(f"Summary saved  : {summary_path}")


if __name__ == "__main__":
    main()

Writing evaluate.py


In [6]:
# Run evaluation (change --num_samples as needed, 0 = all)
!HF_HOME=/content/drive/MyDrive/hf_cache python evaluate.py --device cuda --num_samples 0 --output_dir Evaluation

Using device: cuda
Loading model: Qwen/Qwen2.5-VL-3B-Instruct
preprocessor_config.json: 100% 350/350 [00:00<00:00, 2.83MB/s]
chat_template.json: 1.05kB [00:00, 6.11MB/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
config.json: 1.37kB [00:00, 8.42MB/s]
tokenizer_config.json: 5.70kB [00:00, 3.33MB/s]
vocab.json: 2.78MB [00:00, 126MB/s]
merges.txt: 1.67MB [00:00, 135MB/s]
tokenizer.json: 7.03MB [00:00, 162MB/s]
model.safetensors.index.json: 65.4kB [00:00, 232MB/s]
Fetching 2 files: 100% 2/2 [00:26<00:00, 13.09s/it]
Download complete: 100% 7.51G/7.51G [00:26<00:00, 286MB/s]                
Loading weights: 100% 824/824 [00:02<00:00, 319.66it/s, Materializing param=model.visual.patch_embed.proj.weight]                       

In [7]:
# Preview per-sample results
import pandas as pd
pd.set_option('display.max_colwidth', 80)
df = pd.read_csv('Evaluation/eval_results.csv')
print(f"Mean WER: {df['sample_wer'].astype(float).mean():.4f}")
print(f"Mean CER: {df['sample_cer'].astype(float).mean():.4f}")
df.head(20)

Mean WER: 0.2930
Mean CER: 0.0649


,index,ground_truth,prediction,sample_wer,sample_cer
0,0,put down a resolution on the subject,put down a resolution on the subject,0.0000,0.0000
1,1,and he is to be backed by Mr. Will,and he is to be backed by Mr. Will,0.0000,0.0000
2,2,nominating any more Labour life Peers,nominating any more Labour Life Peers,0.1667,0.0270
3,3,M Ps tomorrow. Mr. Michael Foot has,MPs tomorrow. Mr. Michael Foot has,0.2857,0.0286
4,4,"Griffiths, M P for Manchester Exchange .","Guiffiths, MP for Manchester Exchange.",0.7143,0.0750
5,5,is to be made at a meeting of Labour,is to be made at a meeting of Labour,0.0000,0.0000
6,6,A MOVE to stop Mr. Gaitskell from,A move to stop Mr. Gatsby from,0.2857,0.2727
7,7,0M P for Manchester Exchange .,OM P for Manchester Exchange.,0.5000,0.0667
8,8,A MOVE to stop Mr. Gaitskell from nominating,A MOVE to stop Mr. Gairthell from nominating,0.1250,0.0682
9,9,meeting of Labour 0M Ps tommorow . Mr. Michael,Meeting of Labour 01/7 PR tomorrow. Mr. Michael,0.5556,0.1739


In [8]:
# Print summary
with open('Evaluation/eval_summary.txt') as f:
    print(f.read())

Model: Qwen/Qwen2.5-VL-3B-Instruct
Samples: 10373
WER: 0.2926 (29.26%)
CER: 0.0597 (5.97%)
Time: 3269.6s



In [9]:
# Copy results to Google Drive
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DEST = '/content/drive/MyDrive/Evaluation'
if os.path.exists(DRIVE_DEST):
    shutil.rmtree(DRIVE_DEST)
shutil.copytree('Evaluation', DRIVE_DEST)
print(f'Results saved to Google Drive: {DRIVE_DEST}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results saved to Google Drive: /content/drive/MyDrive/Evaluation
